[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C33_Context_Memory_Course/03_memory/03_memory.ipynb)

# 03 · 文件记忆（file-based memory）

目标：用**纯标准库**从零写出一个文件记忆系统——**结构化条目 → 写入(upsert) → 读取 → 更新 → 去重 → 按需召回 → 淘汰**，全程用确定性逻辑、`assert` 验证，**无需 API key、不写课程目录**（用 in-memory / `tempfile` 后端，可重复运行）。

路线：结构化条目 → 写/读 → upsert(更新而非追加) → 相似度去重 → 按相关性召回(受预算) → 淘汰 → ✏️ 练习 → 📖 答案 → 🧪 真实 memory 工具胶囊。

> 心智模型：**窗口是 RAM、文件记忆是磁盘**（MemGPT）。agent 不必把一切记在窗口里，而是写到「笔记本」上、用时翻一两页回来。难点在记忆这一侧的结构、更新、去重。

## 1 · 结构化记忆条目 + 计数热身

一条记忆 = `{key, value, tags, updated_at}`。**结构化**（而非一坨自由文本）让更新/去重/按类召回/淘汰都可程序化。

先定义条目工厂和一个近似 token 计数器（召回受预算约束时要用；与模块 00/01 同款）。

In [ ]:
import re, math, json, hashlib, tempfile, os

def make_entry(key, value, tags=None, t=0):
    '''构造一条结构化记忆条目。t 是逻辑时间戳(便于淘汰/审计)。'''
    return {'key': key, 'value': value, 'tags': list(tags or []), 'updated_at': t}

def count_tokens(text):
    '''确定性近似 token 计数(真实请用 client.messages.count_tokens)。'''
    if not text:
        return 0
    return sum(max(1, (len(w) + 3) // 4) for w in text.split())

e = make_entry('user.reply_style', '用户偏好简洁、直接的回答', tags=['preference', 'user'], t=1)
print('条目:', e)
print('value 近似 token:', count_tokens(e['value']))
assert set(e) == {'key', 'value', 'tags', 'updated_at'}
assert e['key'] == 'user.reply_style' and 'preference' in e['tags']
assert count_tokens('') == 0 and count_tokens('a b c') >= 3
print('✅ 结构化条目 + 计数就位：每条记忆是有 key 的独立单元，可数、可改、可查')

## 2 · 记忆存储：写入与读取

先做最基本的写入与按 key 读取。后端用一个 `dict`（key → 条目），**可选**落盘到临时文件——重点是逻辑，所以默认 in-memory、可重复运行、不依赖任何固定路径。

In [ ]:
class MemoryStore:
    '''结构化文件记忆的最小内核：dict 后端(key -> entry)，逻辑时钟自增。
       可选 path: 给了就把整库 JSON 落盘(演示『文件』记忆), 不给则纯内存。'''
    def __init__(self, path=None):
        self.entries = {}      # key -> entry
        self.clock = 0         # 逻辑时间戳, 每次写入自增
        self.path = path
    def _tick(self):
        self.clock += 1
        return self.clock
    def _persist(self):
        if self.path:          # 真·写文件(可选): 落盘整库
            with open(self.path, 'w', encoding='utf-8') as f:
                json.dump(self.entries, f, ensure_ascii=False)
    def write(self, key, value, tags=None):
        '''最朴素的写入(暂不去重/淘汰, 下面几节逐步加固)。'''
        self.entries[key] = make_entry(key, value, tags, self._tick())
        self._persist()
        return self.entries[key]
    def read(self, key):
        '''按 key 精确读取, 不存在返回 None。'''
        return self.entries.get(key)
    def all(self):
        return list(self.entries.values())

mem = MemoryStore()
mem.write('user.reply_style', '用户偏好简洁回答', tags=['preference'])
mem.write('project.test', '本项目用 pytest 跑测试', tags=['project'])
print('库里条目数:', len(mem.all()))
print('读 user.reply_style ->', mem.read('user.reply_style')['value'])
print('读 不存在的 key ->', mem.read('nope'))
assert len(mem.all()) == 2
assert mem.read('user.reply_style')['value'] == '用户偏好简洁回答'
assert mem.read('nope') is None
assert mem.read('project.test')['updated_at'] == 2   # 第二个写入, 时钟=2
print('✅ 写入/读取就位：按 key 存取、带逻辑时间戳')

## 3 · upsert：更新而非无脑追加

用户改主意了：先说「简洁」、后说「详细」。**无脑追加**会让库里同时躺着两条矛盾条目，召回时一起进窗口、模型懵掉。

正确做法是 **upsert**（update-or-insert）：写入前按 key 查，命中就**原地更新**(覆盖 value、刷新时间戳)、未命中才新增。顺带得到**幂等**：写两次 == 写一次。

In [ ]:
def upsert(store, key, value, tags=None):
    '''按 key upsert: 命中则更新(覆盖 value, 刷新时间戳, 合并 tags), 否则插入。'''
    existing = store.entries.get(key)
    t = store._tick()
    if existing is not None:
        existing['value'] = value
        existing['updated_at'] = t
        if tags:
            existing['tags'] = sorted(set(existing['tags']) | set(tags))
    else:
        store.entries[key] = make_entry(key, value, tags, t)
    store._persist()
    return store.entries[key]

m = MemoryStore()
upsert(m, 'user.reply_style', '用户偏好简洁回答', tags=['preference'])
upsert(m, 'user.reply_style', '用户改主意了, 想要详细回答', tags=['user'])  # 同 key -> 更新
print('条目数(应为1, 不是2):', len(m.all()))
print('最新 value:', m.read('user.reply_style')['value'])
print('tags 已合并:', m.read('user.reply_style')['tags'])
assert len(m.all()) == 1                                   # 更新而非追加!
assert '详细' in m.read('user.reply_style')['value']         # 取到最新
assert m.read('user.reply_style')['tags'] == ['preference', 'user']  # 合并
# 幂等: 同内容写两次, 结果与写一次一致(条目数不变)
before = len(m.all())
upsert(m, 'user.reply_style', '用户改主意了, 想要详细回答')
assert len(m.all()) == before
print('✅ upsert 就位：同 key 更新而非追加、tags 合并、写两次==写一次(幂等)')

## 4 · 去重：key 规范化 + 相似度阈值

upsert 靠**精确 key** 挡重复。但同一件事常以不同措辞、不同 key 出现(「喜欢简洁」vs「偏好简短直接」)。

**去重**两层把关：① **key 规范化**(小写/去空格/统一分隔符)；② **相似度阈值**(value 与已有条目相似度 ≥ 阈值 -> 判为重复、更新而非新增)。相似度用确定性词频向量 + 余弦(与模块 04 同一套机制)。

In [ ]:
def normalize_key(key):
    '''key 规范化: 小写, 空格/连字符统一成点, 去多余点。
       让 User Reply Style / user-reply-style 落到同一个 key。'''
    k = key.strip().lower()
    k = re.sub(r'[\s\-]+', '.', k)
    k = re.sub(r'[._]+', '.', k).strip('.')
    return k

def embed(text, dim=64):
    '''确定性词频向量(可重复、可断言)。同模块00/04。'''
    vec = [0.0] * dim
    for tok in re.findall(r'[\w]+', text.lower()):
        vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    return vec

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)); nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0

def write_dedup(store, key, value, tags=None, threshold=0.8):
    '''去重写入: 先规范化 key 找精确命中; 再找近似命中(相似度>=阈值);
       命中任一则更新那条, 否则新增。返回 (entry, 动作字符串)。'''
    nkey = normalize_key(key)
    # 1) 精确(规范化后)命中
    if nkey in store.entries:
        upsert(store, nkey, value, tags); return store.entries[nkey], 'update(exact-key)'
    # 2) 近似命中: 与已有条目 value 的最高相似度
    qv = embed(value)
    best_key, best_sim = None, 0.0
    for k, e in store.entries.items():
        s = cosine(qv, embed(e['value']))
        if s > best_sim:
            best_key, best_sim = k, s
    if best_key is not None and best_sim >= threshold:
        upsert(store, best_key, value, tags); return store.entries[best_key], f'update(similar={best_sim:.2f})'
    # 3) 都不命中 -> 新增(用规范化 key)
    upsert(store, nkey, value, tags); return store.entries[nkey], 'insert'

print('规范化:', normalize_key('User Reply Style'), '|', normalize_key('user-reply-style'))
assert normalize_key('User Reply Style') == normalize_key('user-reply-style') == 'user.reply.style'

d = MemoryStore()
_, a1 = write_dedup(d, 'user.style', '用户 喜欢 简洁 回答')
_, a2 = write_dedup(d, 'User Style', '用户 偏好 详细 回答')        # 规范化后同 key -> 更新
_, a3 = write_dedup(d, 'project.test', '本项目 用 pytest 测试')    # 全新 -> 插入
_, a4 = write_dedup(d, 'proj.testing', '本项目 用 pytest 测试')    # value 几乎同 -> 近似命中更新
print('动作序列:', a1, '|', a2, '|', a3, '|', a4)
print('最终条目数(应为2):', len(d.all()))
assert a1 == 'insert' and a2 == 'update(exact-key)'
assert a3 == 'insert' and a4.startswith('update(similar')
assert len(d.all()) == 2          # style 一条 + project 一条, 近似重复被并掉
print('✅ 去重就位：key 规范化挡同义 key、相似度阈值挡近似重复 —— 库不膨胀')

## 5 · 记忆召回：按相关性排序、受预算截取

写得再好用不上也白搭。**召回铁律：受预算约束，只注入最相关的少量，绝不全量回灌**(全量回灌就背叛了文件记忆的初衷)。

召回 = **打分(相关性) → 排序 → 按预算截取**(条数上限 k 且 token ≤ 注入预算)。这套骨架你在模块01(优先级截断)见过、模块04(top-k+注入预算)还会再见。

In [ ]:
def recall(store, query, k=3, inject_budget=40):
    '''召回: 按与 query 的相似度排序, 取前 k 条且累计 token 不超注入预算。
       返回选中的条目列表(少而精, 不是全量)。'''
    qv = embed(query)
    scored = [(cosine(qv, embed(e['value'])), e) for e in store.all()]
    scored.sort(key=lambda x: (-x[0], x[1]['key']))   # 相似度降序, key 稳定
    out, used = [], 0
    for sim, e in scored:
        if len(out) >= k:
            break
        cost = count_tokens(e['value'])
        if used + cost > inject_budget:
            continue                  # 这条放不下, 跳过(看后面更小的)
        out.append(e); used += cost
    return out

store = MemoryStore()
for k, v, tg in [('user.style', '用户 喜欢 简洁 的 回答', ['preference']),
                 ('project.test', '本项目 用 pytest 跑 测试', ['project']),
                 ('deploy.window', '生产 部署 在 周五 晚上', ['fact']),
                 ('lang.pref', '代码 用 python 编写', ['project'])]:
    write_dedup(store, k, v, tg)

hits = recall(store, '怎么 跑 测试', k=2)
print('召回(查询=怎么跑测试):', [e['key'] for e in hits])
assert len(hits) <= 2                              # 受 k 约束
assert hits[0]['key'] == 'project.test'            # 最相关的排第一
assert all(e in store.all() for e in hits)         # 召回的确实来自库
total = sum(count_tokens(e['value']) for e in hits)
assert total <= 40                                 # 受注入预算约束
print('✅ 召回就位：按相关性排序、受 k 与注入预算双重约束、只取少量精华')

## 6 · 淘汰：满了清最旧的

记忆库也有容量。**淘汰**与召回对偶：召回取『最该留的』、淘汰清『最该走的』。最直观的是**按时间淘汰最旧**(`updated_at` 最小)——这正是结构化条目带时间戳的回报。

生产级写入路径三道关：**去重 → upsert → 淘汰**，分别对抗冗余、矛盾、膨胀。

In [ ]:
def write_capped(store, key, value, tags=None, capacity=3, threshold=0.8):
    '''带容量上限的去重写入: 写完若超容量, 按 updated_at 淘汰最旧的。'''
    entry, action = write_dedup(store, key, value, tags, threshold)
    if len(store.entries) > capacity:
        # 按时间淘汰最旧的若干, 直到回到容量
        ordered = sorted(store.entries.values(), key=lambda e: e['updated_at'])
        for old in ordered[:len(store.entries) - capacity]:
            del store.entries[old['key']]
        store._persist()
    return entry, action

cap = MemoryStore()
write_capped(cap, 'a', '事实 甲', capacity=3)
write_capped(cap, 'b', '事实 乙', capacity=3)
write_capped(cap, 'c', '事实 丙', capacity=3)
write_capped(cap, 'd', '事实 丁', capacity=3)   # 超容量 -> 淘汰最旧的 a
keys = sorted(e['key'] for e in cap.all())
print('容量3, 写入4条后剩:', keys)
assert len(cap.all()) == 3                       # 容量守住
assert 'a' not in [e['key'] for e in cap.all()]  # 最旧的 a 被淘汰
assert 'd' in [e['key'] for e in cap.all()]      # 最新的 d 还在
print('✅ 淘汰就位：超容量按时间清最旧 —— 写入三道关(去重/upsert/淘汰)齐了')

---
## ✏️ 练习 1：按 tag 召回（结构化的回报）

结构化条目的好处之一：能**按字段精确取一类**，而非全文回灌。

实现 `recall_by_tag(store, tag, k=3)`：返回 `tags` 含 `tag` 的条目，按 `updated_at` **降序**(新的在前)取前 k 条。

In [ ]:
def recall_by_tag(store, tag, k=3):
    # TODO: 从 store.all() 里筛出 tags 含 tag 的条目,
    #       按 updated_at 降序排序, 取前 k 条返回
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
st = MemoryStore()
for kk, vv, tg in [('p1', '用 pytest', ['project']), ('u1', '喜欢简洁', ['user']),
                   ('p2', '用 ruff 格式化', ['project']), ('p3', 'CI 在 GitHub', ['project'])]:
    write_dedup(st, kk, vv, tg)
proj = recall_by_tag(st, 'project', k=2)
assert len(proj) == 2                              # 受 k 约束
assert all('project' in e['tags'] for e in proj)   # 都含该 tag
assert proj[0]['updated_at'] >= proj[1]['updated_at']  # 新的在前
assert recall_by_tag(st, 'user')[0]['key'] == 'u1'
print('✅ 练习 1 通过：按 tag 精确召回一类、新的优先')

## ✏️ 练习 2：相似度去重判定

去重的核心是「这条 value 和库里某条是不是同一件事」。

实现 `is_duplicate(store, value, threshold=0.8)`：返回 `(是否重复, 命中的key或None, 最高相似度)`。用 `embed` + `cosine` 算 value 与库中每条的相似度，最高的若 ≥ 阈值则判为重复。

In [ ]:
def is_duplicate(store, value, threshold=0.8):
    # TODO: 对 store.all() 每条算 cosine(embed(value), embed(e['value'])),
    #       找最高相似度 best_sim 与对应 best_key;
    #       返回 (best_sim >= threshold, best_key if 重复 else None, best_sim)
    #       空库返回 (False, None, 0.0)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
dup = MemoryStore()
write_dedup(dup, 'a', '用户 喜欢 简洁 直接 的 回答')
is_dup, hit, sim = is_duplicate(dup, '用户 喜欢 简洁 直接 的 回答')   # 完全一样
assert is_dup is True and hit == 'a' and sim > 0.99
is_dup2, hit2, sim2 = is_duplicate(dup, '今天 北京 天气 很 好')        # 毫不相关
assert is_dup2 is False and hit2 is None and sim2 < 0.8
assert is_duplicate(MemoryStore(), 'x')[0] is False                  # 空库
print(f'✅ 练习 2 通过：相似度去重判定(命中 sim={sim:.2f}, 无关 sim={sim2:.2f})')

## ✏️ 练习 3：召回并组装进上下文

把召回接到「组装上下文」：召回相关记忆，拼成一段可注入系统提示的文本块。

实现 `build_memory_block(store, query, k=3, inject_budget=40)`：用 `recall` 取条目，拼成形如 `'相关记忆:\n- <value>\n- <value>'` 的字符串；无召回则返回 `''`(空串, 不注入)。

In [ ]:
def build_memory_block(store, query, k=3, inject_budget=40):
    # TODO: hits = recall(store, query, k, inject_budget)
    #       若 hits 为空 -> 返回 ''
    #       否则返回 '相关记忆:\n' + '\n'.join('- ' + e['value'] for e in hits)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
s3 = MemoryStore()
for kk, vv in [('t', '本项目 用 pytest 测试'), ('d', '周五 晚上 部署')]:
    write_dedup(s3, kk, vv)
block = build_memory_block(s3, '怎么 测试', k=1)
print(repr(block))
assert block.startswith('相关记忆:')
assert 'pytest' in block and block.count('- ') == 1     # k=1 只召回一条
assert build_memory_block(MemoryStore(), '任何') == ''   # 空库 -> 不注入
print('✅ 练习 3 通过：召回 -> 组装成可注入的记忆块、空库不注入')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def recall_by_tag(store, tag, k=3):
    hits = [e for e in store.all() if tag in e['tags']]
    hits.sort(key=lambda e: -e['updated_at'])
    return hits[:k]

In [ ]:
# 练习 2 参考答案
def is_duplicate(store, value, threshold=0.8):
    qv = embed(value)
    best_key, best_sim = None, 0.0
    for e in store.all():
        s = cosine(qv, embed(e['value']))
        if s > best_sim:
            best_key, best_sim = e['key'], s
    dup = best_sim >= threshold
    return dup, (best_key if dup else None), best_sim

In [ ]:
# 练习 3 参考答案
def build_memory_block(store, query, k=3, inject_budget=40):
    hits = recall(store, query, k, inject_budget)
    if not hits:
        return ''
    return '相关记忆:\n' + '\n'.join('- ' + e['value'] for e in hits)

---
## 🧪 真实数据胶囊：MEMORY.md 与真实记忆条目的形状

真实 agent 的文件记忆，最常见的载体就是一份 **MEMORY.md** + 若干结构化笔记。下面用一段**贴近真实**的 MEMORY.md(偏好/项目约定/事实)，解析成本课的结构化条目，再用上面写的 `recall` 召回——体会「真实记忆文件 ↔ 本课条目库」的对应。

> 形状对照：真实里这份文件由人或 Claude 的 memory 工具维护；本课把它解析成 `{key,value,tags,updated_at}` 条目来管理。

In [ ]:
# 一段贴近真实的 MEMORY.md(Markdown 小节 + 要点)
MEMORY_MD = '''# MEMORY
## user
- reply_style: 用户偏好简洁、直接的回答, 少寒暄
- timezone: 用户在 Asia/Shanghai
## project
- test: 本项目用 pytest 跑测试, 覆盖率要 > 80%
- style: 代码用 ruff 格式化
## fact
- deploy: 生产部署在每周五晚上
'''

def parse_memory_md(text):
    '''把 MEMORY.md 解析成结构化条目: ## 小节名当 tag, '- key: value' 当条目。'''
    store = MemoryStore()
    section = 'misc'
    for ln in text.splitlines():
        ln = ln.strip()
        if ln.startswith('## '):
            section = ln[3:].strip()
        elif ln.startswith('- ') and ':' in ln:
            k, v = ln[2:].split(':', 1)
            write_dedup(store, f'{section}.{k.strip()}', v.strip(), tags=[section])
    return store

real = parse_memory_md(MEMORY_MD)
print('解析出的条目 key:', sorted(e['key'] for e in real.all()))
print('按 project 召回:', [e['key'] for e in recall_by_tag(real, 'project')] if 'recall_by_tag' in dir() else '(先做练习1)')
hits = recall(real, 'pytest 测试 覆盖率', k=1)   # 命中 project.test 的 pytest 词
assert len(real.all()) == 5
assert hits[0]['key'] == 'project.test'         # 真实记忆里召回正确
assert any('user' in e['tags'] for e in real.all())
print('✅ 真实 MEMORY.md 解析成条目库、召回正确 —— 本课内核直接适用')

**🧪 胶囊练习**：实现 `memory_stats(store)`：返回 `{'total': 条目数, 'by_tag': {tag: 数量}}`。(真实里统计『记忆库有多大、各类各多少』就是这么做的，是淘汰/体检的基础。)

In [ ]:
def memory_stats(store):
    # TODO: total = 条目总数; by_tag = 每个 tag 出现的条目数(一条可含多 tag)
    raise NotImplementedError

In [ ]:
# 自测
stats = memory_stats(real)
assert stats['total'] == 5
assert stats['by_tag']['project'] == 2 and stats['by_tag']['user'] == 2
assert stats['by_tag']['fact'] == 1
print('记忆体检:', stats)
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def memory_stats(store):
    by_tag = {}
    for e in store.all():
        for t in e['tags']:
            by_tag[t] = by_tag.get(t, 0) + 1
    return {'total': len(store.all()), 'by_tag': by_tag}

---
## 🔧 旁注：对应的真实 Anthropic memory 工具长什么样

本课从零写的记忆系统，对应真实里的 **Claude memory 工具**(`memory_20250818`)：模型获得一个 `/memories` 目录和一组操作(`view` / `create` / `str_replace` / `insert` / `delete`)，**由模型自己决定**记什么、何时记、记到哪、何时改——你只需在客户端实现这些文件操作的后端(就是本课的 `MemoryStore`)。

```python
import anthropic
client = anthropic.Anthropic()                  # 读 ANTHROPIC_API_KEY
resp = client.messages.create(
    model='claude-sonnet-4-6', max_tokens=1024,
    tools=[{'type': 'memory_20250818', 'name': 'memory'}],   # 声明 memory 工具
    messages=[{'role': 'user', 'content': '记住我喜欢简洁的回答'}],
)
# 模型会发出 memory 工具调用(如 create /memories/user.md), 你的后端执行文件读写
# —— 后端逻辑就是本课的 MemoryStore: write/read/upsert/dedup/淘汰
```

对应关系：你的 `MemoryStore.write/read/upsert` ↔ memory 工具的 `create/view/str_replace`、你的去重/淘汰是**真实工具不强制、但生产必须自己加**的一层。这就是「scaffold 可迁移」——而且**无 key 时本课全程用确定性逻辑跑通，绝不阻断**。

> ⚠️ **安全提醒**(真实接口同样适用)：绝不要把 API key、密码、PII 写进记忆；多用户系统要按用户隔离记忆目录并鉴权。校验模型给的路径、禁止 `..` 越界——记忆持久化带来隐私与合规责任。

### 小结
- **窗口是 RAM、文件记忆是磁盘**(MemGPT)：把该长期记住的移出窗口、按需召回少量 —— 突破窗口上限的根本手段。
- **更新而非追加**(upsert)：同 key 原地更新、幂等;无脑追加会膨胀 + 留矛盾副本。
- **去重**：key 规范化 + 相似度阈值挡近似重复 —— 否则污染召回、浪费 token。
- **结构化条目** {key,value,tags,updated_at}：让更新/去重/按类召回/淘汰全可程序化(自由文本做不到)。
- **召回**：打分→排序→按 k 与注入预算截取, 只取少量精华、绝不全量回灌。
- **写入三道关**：去重 → upsert → 淘汰, 分别对抗冗余/矛盾/膨胀。

下一站：**模块 04 · 检索入上下文** —— 把『按相关性召回』从记忆推广到海量外部知识，即 agent 的 RAG。